# Calli 笔画阶段与结体部件：YOLO26s 实例分割

本笔记本用于你在 **Google Colab 或 VS Code 连接的 Colab 内核**中自行训练。模型同时输出原始笔画阶段和结体部件的类别、框与轮廓；不接入 Fontify。

直接使用 Calli 原始标注，每个阶段 annotation 是一个独立实例。类别表为 50 个 BF 笔画阶段类别＋20 个 JT 结体类别，共 70 类；程序从 data.yaml 和 classes.json 动态读取类别。

**执行顺序：**准备远端文件 → 安装依赖 → 核对数据 → 显式运行训练 → 验证与推理。只打开此文件不会训练。VS Code 本地文件不会自动出现在 Colab 内核里，需要上传 `yolo` 文件夹或压缩包。

建议选择 GPU 运行时。默认 640 输入、batch 8、100 epoch、patience 25；显存不足时先把 batch 改为 4。

## 1. 把 yolo 文件夹送到 Colab

推荐把完整 `yolo/` 文件夹或包含顶层 `yolo/` 的压缩包上传到 Google Drive 的 `MyDrive/CalliYOLO/`。下面把代码和数据复制到 Colab 本地磁盘以加快读取，模型保存在 Drive。

也可以设置 `STORAGE_MODE="upload"` 上传压缩包；该模式的输出暂存在 `/content/yolo/runs`，断开临时运行时前需自行下载或复制到 Drive。

In [ ]:
from pathlib import Path
import shutil
import zipfile

STORAGE_MODE = "drive"  # "drive" 或 "upload"
ROOT = Path("/content/yolo")
DRIVE_HOME = Path("/content/drive/MyDrive/CalliYOLO")
ARCHIVE_NAME = "yolo_colab.zip"  # 与实际压缩包文件名一致

if STORAGE_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS = DRIVE_HOME / "runs"
    if not (ROOT / "dataset/data.yaml").is_file():
        archive = DRIVE_HOME / ARCHIVE_NAME
        if archive.is_file():
            with zipfile.ZipFile(archive) as bundle:
                bundle.extractall(ROOT.parent)
        elif (DRIVE_HOME / "yolo/dataset/data.yaml").is_file():
            shutil.copytree(DRIVE_HOME / "yolo", ROOT, dirs_exist_ok=True)
        else:
            raise FileNotFoundError(
                f"请先上传完整 yolo 文件夹或 {archive}；压缩包顶层应为 yolo/。"
            )
elif STORAGE_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("请上传一个包含顶层 yolo/ 的 ZIP。")
    with zipfile.ZipFile(archives[0]) as bundle:
        bundle.extractall(ROOT.parent)
    RUNS = ROOT / "runs"
else:
    raise ValueError("STORAGE_MODE 只能是 drive 或 upload")

assert (ROOT / "dataset/data.yaml").is_file(), "没有找到 yolo/dataset/data.yaml"
assert (ROOT / "train.py").is_file(), "没有找到 yolo/train.py"
RUNS.mkdir(parents=True, exist_ok=True)
print("代码和数据：", ROOT)
print("模型输出：", RUNS)

## 2. 安装依赖并检查 GPU

依赖固定 Ultralytics 版本。首次真正训练时会下载 COCO 初始化权重；该初始化权重尚不能识别 Calli 笔画阶段和部件。

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")], check=True)
import torch
import ultralytics
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "未启用 GPU")
if not torch.cuda.is_available():
    raise RuntimeError("请选择 GPU 运行时后重新连接内核。")

## 3. 核对数据（不加载模型，不训练）

检查图片/标签配对、类别范围、归一化轮廓以及 split 路径。数据构建时的同源字形分组信息见数据集附带的 manifest/summary 文件。

In [ ]:
subprocess.run([sys.executable, str(ROOT / "train.py"), "--dry-run"], check=True, cwd=ROOT)
print((ROOT / "dataset/data.yaml").read_text(encoding="utf-8"))

## 4. 开始训练

这一格才会进行参数更新。`best.pt` 按验证表现保存，`last.pt` 用于断点续训，输出目录名重复时 Ultralytics 会顺延编号。默认关闭翻转、Mosaic、MixUp、Copy-paste，并保留可重叠实例 mask。

若仅想检查训练管线，可把 `EPOCHS` 临时设为 1，并换一个 `RUN_NAME`；这不是正式识别效果。

In [ ]:
EPOCHS = 100
BATCH = 8
IMGSZ = 640
RUN_NAME = "calli_phase_jieti_yolo26s"

command = [
    sys.executable, str(ROOT / "train.py"),
    "--model", "yolo26s-seg.pt", "--epochs", str(EPOCHS),
    "--batch", str(BATCH), "--imgsz", str(IMGSZ),
    "--device", "0", "--project", str(RUNS), "--name", RUN_NAME,
]
subprocess.run(command, check=True, cwd=ROOT)

## 5. 加载实际训练权重并验证

以下选择本次名称前缀下最近保存的 `best.pt`，先打印完整路径供核对。若要评价指定运行，直接修改 `BEST_WEIGHT`。验证指标只代表当前原始阶段／结体标注和当前划分的表现。

In [ ]:
import yaml
from ultralytics import YOLO
sys.path.insert(0, str(ROOT))
from train import read_dataset

candidates = list(RUNS.glob(f"{RUN_NAME}*/weights/best.pt"))
if not candidates:
    raise FileNotFoundError("尚无训练后的 best.pt；先运行训练或指定已有权重。")
BEST_WEIGHT = max(candidates, key=lambda path: path.stat().st_mtime)
print("评价权重：", BEST_WEIGHT)
resolved = read_dataset(ROOT / "dataset/data.yaml")
RUNTIME_DATA = ROOT / "dataset_colab_resolved.yaml"
RUNTIME_DATA.write_text(yaml.safe_dump(resolved, allow_unicode=True, sort_keys=False), encoding="utf-8")
model = YOLO(str(BEST_WEIGHT))
metrics = model.val(data=str(RUNTIME_DATA), split="val", device=0, imgsz=IMGSZ,
                    project=str(RUNS), name="validation")
print(metrics.results_dict)

## 6. 导出预测图与轮廓 JSON

默认预测验证集目录，可把 `SOURCE` 改成你上传的某个汉字图片路径。JSON 包含类名、中文名、置信度、边界框和像素/归一化轮廓；叠加图使用模型类名，避免运行时缺少中文字体。

每个实例还导出 `mask_rle`（COCO RLE），以无损保留预测二值 mask 的不连通部分与孔洞；需要精确像素区域时请使用它。`polygon_xy`／`polygon_normalized` 只是便于使用的轮廓表达，可能连接原本分开的区域或忽略孔洞。

In [ ]:
from datetime import datetime
from IPython.display import display, Image

SOURCE = ROOT / "dataset/images/val"
PREDICTIONS = RUNS / ("predictions_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
subprocess.run([
    sys.executable, str(ROOT / "predict.py"), "--weights", str(BEST_WEIGHT),
    "--source", str(SOURCE), "--output", str(PREDICTIONS), "--device", "0",
], check=True, cwd=ROOT)
for path in sorted(PREDICTIONS.glob("*.png"))[:4]:
    display(Image(filename=str(path), width=640))
print("预测结果：", PREDICTIONS)

## 7. 可选：断点续训

断线后先重新执行文件准备、安装依赖、数据检查，再填写原运行的 `last.pt`。恢复会沿用 checkpoint 内的训练计划，不能用 `--epochs` 把已完成的训练直接当作未完成任务恢复。需要额外微调时，另起运行并用旧权重作为 `--model`。

In [ ]:
RUN_RESUME = False
LAST_WEIGHT = RUNS / "calli_phase_jieti_yolo26s/weights/last.pt"  # 修改为实际运行目录
if RUN_RESUME:
    subprocess.run([
        sys.executable, str(ROOT / "train.py"), "--resume", str(LAST_WEIGHT),
        "--project", str(RUNS), "--device", "0",
    ], check=True, cwd=ROOT)
else:
    print("未执行续训。填写 LAST_WEIGHT 并把 RUN_RESUME 改为 True 后再运行此格。")